In [1]:
import os
import pandas as pd
import numpy as np

In [2]:
Dir_github = r'X:\PROJECTS\ABC@UofSC\3-Experiments\NSF_valence\github\nsf_valence'
Dir_ratings = os.path.join(Dir_github,'derivatives','behavioral','ratings','text')
df_ratings = pd.read_csv(os.path.join(Dir_ratings,'ratings_text_att2_comp2.csv'), dtype={'idx': str,'ethnicity':str},keep_default_na=False)


In [3]:
df_ratings.loc[df_ratings['segID']==1].groupby(['group']).count()

,subID,idx,text,text_idx,segID,V,A,RT,age,sex,age_prescreen,sex_prescreen,attention_correct,attention_correct%,comp_correct,comp_correct%,source,ethnicity,residence_states
group,,,,,,,,,,,,,,,,,,,
middle,352,352,352,352,352,352,352,352,352,352,352,352,352,352,352,352,352,352,352
old,350,350,350,350,350,350,350,350,350,350,350,350,350,350,350,350,350,350,350
young,377,377,377,377,377,377,377,377,377,377,377,377,377,377,377,377,377,377,377


In [5]:
df_ratings_bysub = df_ratings.loc[df_ratings['segID']==1].groupby(['subID','group']).count().reset_index()
print(f"On average, each participant rated {df_ratings_bysub['text'].mean().round(2)} narratives, with a range of [{df_ratings_bysub['text'].min()},{df_ratings_bysub['text'].max()}].")
df_ratings_bysub.groupby(['group']).mean(numeric_only=True).round(2)['text']

On average, each participant rated 3.46 narratives, with a range of [1,15].


group
middle    3.09
old       4.79
young     3.02
Name: text, dtype: float64

In [6]:
#### summarize ethnicity
df_demographic = df_ratings.drop_duplicates(subset=['subID'],keep='first')

print(f"The final sample consisted of {df_demographic.shape[0]} participants.")
for group in ['young','middle','old']:
    df = df_demographic.loc[df_demographic['group']==group].copy()
    print(f"----- {group} N = {df.shape[0]} -----")
    print(f"Among {group} adults, the mean age was {df['age'].mean().round(2)} years (SD = {np.round(df['age'].std(),2)}),")
    print(f"with {df.loc[df['sex']=='female'].shape[0]} females and {df.loc[df['sex']=='male'].shape[0]} males.")
    print('Ethnicity distribution was as follows: ')
    for ethnicity in ['White','Black','Asian','Other','NA']:
        
        N_group = df.shape[0]
        n_ethnicity = df.loc[df_ratings['ethnicity']==ethnicity].shape[0]
        # print(f"{n_ethnicity} ({np.round(n_ethnicity/N_group*100,2)}%) as {ethnicity}")
        print(f"{ethnicity} (n = {n_ethnicity}, {np.round(n_ethnicity/N_group*100,2)}%),")


The final sample consisted of 312 participants.
----- young N = 125 -----
Among young adults, the mean age was 28.04 years (SD = 6.82),
with 77 females and 48 males.
Ethnicity distribution was as follows: 
White (n = 48, 38.4%),
Black (n = 38, 30.4%),
Asian (n = 3, 2.4%),
Other (n = 2, 1.6%),
NA (n = 32, 25.6%),
----- middle N = 114 -----
Among middle adults, the mean age was 48.32 years (SD = 5.24),
with 67 females and 47 males.
Ethnicity distribution was as follows: 
White (n = 68, 59.65%),
Black (n = 32, 28.07%),
Asian (n = 3, 2.63%),
Other (n = 5, 4.39%),
NA (n = 6, 5.26%),
----- old N = 73 -----
Among old adults, the mean age was 65.12 years (SD = 3.66),
with 53 females and 20 males.
Ethnicity distribution was as follows: 
White (n = 51, 69.86%),
Black (n = 19, 26.03%),
Asian (n = 0, 0.0%),
Other (n = 1, 1.37%),
NA (n = 2, 2.74%),


#### Calculate the segment-level mean ratings by group and across groups

In [21]:
df_ratings_groupmean = df_ratings.groupby(['group','text','segID']).mean(numeric_only=True).reset_index()
df_ratings_grandmean = df_ratings.groupby(['text','segID']).mean(numeric_only=True).reset_index()
df_ratings_grandmean['group'] = 'all'
df_ratings_mean = pd.concat([df_ratings_groupmean,df_ratings_grandmean],ignore_index=True)
df_ratings_mean

,group,text,segID,text_idx,V,A,RT,age,attention_correct,attention_correct%,comp_correct,comp_correct%
0,middle,AgedMother,1,1.958333,0.477344,3.900781,8.137708,48.125000,2.958333,0.986111,4.625000,0.925000
1,middle,AgedMother,2,1.958333,0.860156,4.084375,4.805917,48.125000,2.958333,0.986111,4.625000,0.925000
2,middle,AgedMother,3,1.958333,-0.435156,3.846875,5.093167,48.125000,2.958333,0.986111,4.625000,0.925000
3,middle,AgedMother,4,1.958333,1.309375,4.106250,5.427708,48.125000,2.958333,0.986111,4.625000,0.925000
4,middle,AgedMother,5,1.958333,2.075000,4.526562,3.187500,48.125000,2.958333,0.986111,4.625000,0.925000
...,...,...,...,...,...,...,...,...,...,...,...,...
11873,all,UserFriendly,137,2.086957,1.492255,4.670788,2.228522,46.695652,2.934783,0.978261,3.021739,0.604348
11874,all,UserFriendly,138,2.086957,1.065897,4.962228,5.225870,46.695652,2.934783,0.978261,3.021739,0.604348
11875,all,UserFriendly,139,2.086957,1.038587,5.083288,3.343413,46.695652,2.934783,0.978261,3.021739,0.604348
11876,all,UserFriendly,140,2.086957,1.980571,5.228804,4.968652,46.695652,2.934783,0.978261,3.021739,0.604348


In [25]:
### add the segment texts and timestamps in the audio recordings
Dir_timestamps = os.path.join(Dir_github,'derivatives','behavioral','segmentation')

dfs_timestamps = []
for text in df_ratings_mean['text'].unique():
    path_file = os.path.join(Dir_timestamps,f'segmentation_{text}.csv')
    df_timestamps = pd.read_csv(path_file)
    dfs_timestamps.append(df_timestamps)
dfs_timestamps = pd.concat(dfs_timestamps,ignore_index=True)

df_ratings_mean = pd.merge(left = df_ratings_mean, left_on = ['text','segID'],
         right = dfs_timestamps, right_on = ['narrative','segID'],
         how = 'left')

df_ratings_mean.to_csv(os.path.join(Dir_github,'derivatives','behavioral','ratings','text','ratings_text_att2_comp2_mean.csv'),index=False)